In [ ]:
## Ethical Risk Assessment of Predictive AI Models in Healthcare Logistics

This notebook builds and evaluates predictive models that flag **ICU capacity risk** using facility-level hospital capacity reporting.

### Why this matters (ethical + operational)
In healthcare logistics, predictive signals can influence resource allocation and escalation decisions. That makes it critical to assess:
- **Reliability**: Does the model perform well overall?
- **Equity**: Does performance differ meaningfully across **states** (a proxy for regional context)?
- **Transparency**: Can we explain the main drivers of predictions (e.g., occupancy rates)?

### Constraints
- Load only **100,000 rows** for speed.
- Use clean preprocessing, safe division (avoid divide-by-zero), and reproducible splits.


In [ ]:
# -----------------------------
# 1) Imports and configuration
# -----------------------------

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import xgboost as xgb
import shap

RANDOM_STATE = 42
SAMPLE_ROWS = 100_000

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

DATA_PATH = "../data/COVID-19_Reported_Patient_Impact_and_Hospital_Capacity_by_Facility_--_RAW_20260407.csv"


In [ ]:
# -----------------------------
# 2) Data loading (100,000 rows)
# -----------------------------

# Load only a sample (first N rows) for faster processing.
# Using nrows avoids scanning the full file just to sample.

df_raw = pd.read_csv(DATA_PATH, nrows=SAMPLE_ROWS)

print("Loaded shape:", df_raw.shape)
df_raw.head()

## Data understanding

We’ll inspect the dataset structure and confirm the presence (or nearest equivalents) of the required columns:

Required canonical columns:
- `total_beds`, `inpatient_beds_used`
- `total_icu_beds`, `icu_beds_used`
- `critical_staffing_shortage_today`
- `state`, `collection_week`


In [ ]:
# First 5 rows
print("First 5 rows:")
display(df_raw.head(5))

# Dataset info
print("\nInfo:")
display(df_raw.info())

# Summary statistics (numeric)
print("\nSummary statistics:")
display(df_raw.describe(include=[np.number]))

# Column names
print("\nColumn names:")
print(list(df_raw.columns))

## Data preprocessing

### Column selection + canonical renaming
The raw file uses `*_7_day_avg` column names for bed and ICU metrics. To keep the project consistent with the requested schema, we:
- **Select** the nearest available raw columns
- **Rename** them into canonical names (`total_beds`, `icu_beds_used`, etc.)

### Missing values
We drop rows with missing values in the selected fields.


In [ ]:
# Map raw column names -> canonical column names required by the project
# If a required column is not present, we create a safe fallback.

required_canonical = [
    "total_beds",
    "inpatient_beds_used",
    "total_icu_beds",
    "icu_beds_used",
    "critical_staffing_shortage_today",
    "state",
    "collection_week",
]

column_map_candidates = {
    # canonical: candidate raw columns in order of preference
    "total_beds": ["total_beds", "total_beds_7_day_avg", "total_beds_7_day_sum"],
    "inpatient_beds_used": ["inpatient_beds_used", "inpatient_beds_used_7_day_avg", "inpatient_beds_used_7_day_sum"],
    "total_icu_beds": ["total_icu_beds", "total_icu_beds_7_day_avg", "total_icu_beds_7_day_sum"],
    "icu_beds_used": ["icu_beds_used", "icu_beds_used_7_day_avg", "icu_beds_used_7_day_sum"],
    "critical_staffing_shortage_today": ["critical_staffing_shortage_today"],
    "state": ["state"],
    "collection_week": ["collection_week"],
}

resolved = {}
missing = []
for canonical, candidates in column_map_candidates.items():
    found = next((c for c in candidates if c in df_raw.columns), None)
    if found is None:
        missing.append(canonical)
    else:
        resolved[canonical] = found

print("Resolved column mapping (canonical -> raw):")
for k, v in resolved.items():
    print(f"- {k}: {v}")

if missing:
    print("\nCanonical columns not present in raw data:")
    for m in missing:
        print(f"- {m}")

# Build the working dataframe with selected fields
selected_raw_cols = [resolved[c] for c in resolved.keys()]
df = df_raw[selected_raw_cols].copy()

# Rename to canonical names
rename_map = {raw: canonical for canonical, raw in resolved.items()}
df = df.rename(columns=rename_map)

# If staffing shortage is missing, create a conservative placeholder (0) and document it.
if "critical_staffing_shortage_today" not in df.columns:
    df["critical_staffing_shortage_today"] = 0

# Enforce expected columns ordering (where available)
for col in required_canonical:
    if col not in df.columns:
        df[col] = np.nan

df = df[required_canonical]

print("\nWorking shape before dropna:", df.shape)

# Drop missing values in required columns
before = df.shape[0]
df = df.dropna()
after = df.shape[0]
print(f"Rows dropped due to missing values: {before - after:,} (remaining: {after:,})")

df.head()

## Feature engineering

We create operational features and a binary target:
- `icu_occupancy_rate = icu_beds_used / total_icu_beds`
- `bed_occupancy_rate = inpatient_beds_used / total_beds`

Target:
- `ICU_capacity_risk = 1` if `icu_occupancy_rate >= 0.90` else `0`

Division is done safely to avoid divide-by-zero.
